In [1]:
import pandas as pd
import numpy as np
import os

# Mapping Setup

In [6]:
BASE_PATH = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/"
OUT_PATH = "/storage/Arushi/090526_EvoAge/kg_formation/processed_data/"

In [7]:
# ── 1. UniProt ────────────────────────────────────────────────────────────────
# Full UniProt ↔ NCBI mapping table (used for cross-referencing protein/gene IDs)
Uniprot_names = pd.read_csv(f'{BASE_PATH}databases_for_mapping/uniprot/UNIPROT_NCBI_ALL_MAPPED.csv')

# UniProt Accession → Recommended Protein Name (RecName) mapping
# RecName is the curated "recommended name" from UniProt's controlled vocabulary
Uniprot_Recname = pd.read_csv(f'{BASE_PATH}databases_for_mapping/uniprot/Uniprot_ID_Recname.csv')
Uniprot_Recname_dict = dict(zip(Uniprot_Recname['AC'], Uniprot_Recname['RecName']))
# e.g. {'P04637': 'Cellular tumor antigen p53', ...}


/tmp/ipykernel_2627425/1746856877.py:3: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  Uniprot_names = pd.read_csv(f'{BASE_PATH}databases_for_mapping/uniprot/UNIPROT_NCBI_ALL_MAPPED.csv')


In [8]:
# ── 2. Tissue Ontology (BTO) ──────────────────────────────────────────────────
# BTO (BRENDA Tissue Ontology): maps tissue/cell-type names → BTO IDs
# Used to normalize free-text tissue names into ontology-controlled IDs
BTO = pd.read_csv(f'{BASE_PATH}databases_for_mapping/bto/BTO_ALL_COMBINED.csv')
BTO_dict = dict(zip(BTO['name'], BTO['ID']))
# e.g. {'liver': 'BTO:0000759', 'blood plasma': 'BTO:0000131', ...}

In [9]:
# # ── 3. PubChem ────────────────────────────────────────────────────────────────

# # 3a. CID-Synonym filtered file: maps chemical synonyms/names → PubChem CIDs
# #     Columns: [0] = CID (int), [1] = synonym (string)
# #     No header in this file, hence header=None
# Pubchem_Syn_fil = pd.read_csv(f'{BASE_PATH}databases_for_mapping/pubchem/CID-Synonym-filtered',
#     sep='\t',
#     header=None
# )

# # Sanity checks (run interactively to verify mappings are loading correctly)
# # Pubchem_Syn_fil[Pubchem_Syn_fil[1] == 'metformin']  # check metformin CID
# # Pubchem_Syn_fil[Pubchem_Syn_fil[0] == 4091]          # check CID 4091 synonyms

# # synonym → CID lookup dict (lowercased keys for case-insensitive matching)
# # Note: if a synonym maps to multiple CIDs, last one wins (dict deduplication)
# Pubchem_Syn_fil_dict       = dict(zip(Pubchem_Syn_fil[1], Pubchem_Syn_fil[0]))          # original case
# Pubchem_Syn_fil_dict_lower = dict(zip(Pubchem_Syn_fil[1].str.lower(), Pubchem_Syn_fil[0]))  # lowercase for fuzzy match


# # 3b. Combined PubChem compound table: CID → IUPAC name + SMILES
# #     Pre-built pickle for fast loading (large file)
# Pubchem = pd.read_pickle(f'{BASE_PATH}databases_for_mapping/pubchem/combined_df.pkl')

# # CID → canonical SMILES (used for chemical structure representation in KG)
# Pubchem_CID_Smile_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_SMILES']))

# # CID → IUPAC name (used as Head_detail_name for ChemicalEntity nodes)
# Pubchem_IUPAC_CID_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_IUPAC_NAME']))

In [10]:
# ── 4. Ensembl → NCBI Gene Symbol Mapping ────────────────────────────────────
# Raw table: 86,402 Ensembl IDs mapped to NCBI gene symbols
# 'name' = NCBI gene symbol, 'initial_alias' = Ensembl ID
ENS_2NCBI = pd.read_csv(f'{BASE_PATH}databases_for_mapping/ENSEMBL/ENSEMBLE_ID_2_NCBI_Gene_SYM.csv')

# Drop rows with no gene symbol — leaves ~42,530 valid mappings
ENS_2NCBI = ENS_2NCBI[~ENS_2NCBI['name'].isna()]

# Gene Symbol → Ensembl ID lookup dict
# Used to annotate NCBI_ALL_GENE with Ensembl IDs below
NCBI_2ENS__dict = dict(zip(ENS_2NCBI['name'], ENS_2NCBI['initial_alias']))

# Free memory — large file, not needed beyond this mapping
del ENS_2NCBI

In [11]:
# ── 5. Human NCBI Gene Info ───────────────────────────────────────────────────
# Official NCBI gene_info file for Homo sapiens
# Columns of interest: GeneID, Symbol, Synonyms, description
NCBI_ALL_GENE = pd.read_csv(f'{BASE_PATH}databases_for_mapping/ncbi/Homo_sapiens.gene_info',
    sep='\t'
)

# Annotate each gene with its Ensembl ID (via Symbol → Ensembl dict built above)
NCBI_ALL_GENE['ENSEMBLE_ID'] = NCBI_ALL_GENE['Symbol'].map(NCBI_2ENS__dict)

# GeneID → gene description  (used as Head/Tail_detail_name for Gene nodes)
NCBI_ALL_GENEname_dict    = dict(zip(NCBI_ALL_GENE['GeneID'], NCBI_ALL_GENE['description']))

# GeneID → gene Symbol       (used to convert numeric IDs back to readable symbols)
NCBI_ALL_GENEIDname_dict  = dict(zip(NCBI_ALL_GENE['GeneID'], NCBI_ALL_GENE['Symbol']))

# Gene Symbol → gene description  (direct symbol-based lookup, no ID needed)
NCBI_ALL_Symb_Desc_dict   = dict(zip(NCBI_ALL_GENE['Symbol'], NCBI_ALL_GENE['description']))

# Synonym → canonical Gene Symbol (pipe-delimited field in NCBI, e.g. "TP53|P53|LFS1")
# Step 1: Build raw dict — keys are the full pipe-joined synonym strings
NCBI_ALL_GENE_Syn_Dict = dict(zip(NCBI_ALL_GENE['Synonyms'], NCBI_ALL_GENE['Symbol']))

# Step 2: Explode pipe-delimited synonyms so each alias gets its own entry
# e.g. "TP53|P53|LFS1" → {'TP53': 'TP53', 'P53': 'TP53', 'LFS1': 'TP53'}
# Used to recover canonical symbols from non-standard gene name variants
exploded_dict_NCBI_ALL_GENE_Syn_Dict = {}
for k, v in NCBI_ALL_GENE_Syn_Dict.items():
    for single_key in k.split('|'):
        exploded_dict_NCBI_ALL_GENE_Syn_Dict[single_key.strip()] = v

In [160]:

NCBI_cele_gene = pd.read_csv(f'{BASE_PATH}databases_for_mapping/ncbi/Caenorhabditis_elegans.gene_info',
    sep='\t'
)
NCBI_cele_gene

NCBI_Cele_SYM_Descrip_dict = dict(zip(NCBI_cele_gene['Symbol_from_nomenclature_authority'], NCBI_cele_gene['description']))
NCBI_Cele_SYM_Descrip_dict

{'homt-1': 'Alpha N-terminal protein methyltransferase 1',
 'nlp-40': 'Peptide P4',
 'rcor-1': 'REST corepressor rcor-1',
 'sesn-1': 'Sestrin homolog',
 'pgs-1': 'CDP-diacylglycerol--glycerol-3-phosphate 3-phosphatidyltransferase',
 'Y48G1C.5': 'Tyrosine-protein phosphatase domain-containing protein',
 'Y48G1C.6': 'HTH CENPB-type domain-containing protein',
 'pid-2': 'Protein pid-2',
 'rab-11.1': 'Ras-related protein rab-11.1',
 'rpl-7': 'Large ribosomal subunit protein uL30',
 'F53G12.9': 'GST_C_6 domain-containing protein;Metaxin glutathione S-transferase domain-containing protein',
 'F53G12.8': 'Uncharacterized protein',
 'col-45': 'Collagen triple helix repeat protein;Nematode cuticle collagen N-terminal domain-containing protein',
 'spe-8': 'Spermatocyte protein spe-8',
 'mex-3': 'Growth-regulating factor;K Homology domain-containing protein',
 'bli-3': 'Dual oxidase 1',
 'ptr-11': 'SSD domain-containing protein',
 'cest-27': 'Carboxylic ester hydrolase',
 'F56C11.3': 'Sulfhydryl 

In [177]:

NCBI_droso_gene = pd.read_csv(f'{BASE_PATH}databases_for_mapping/ncbi/Danio_rerio.gene_info',
    sep='\t'
)
NCBI_droso_gene

NCBI_droso_SYM_Descrip_dict = dict(zip(NCBI_droso_gene['Symbol'], NCBI_droso_gene['description']))
NCBI_droso_SYM_Descrip_dict


NCBI_saccharo_gene = pd.read_csv(f'{BASE_PATH}databases_for_mapping/ncbi/Saccharomyces_cerevisiae.gene_info',sep='\t')
NCBI_saccharo_gene

NCBI_saccharo_SYM_Descrip_dict = dict(zip(NCBI_saccharo_gene['Symbol'], NCBI_saccharo_gene['description']))
# NCBI_saccharo_SYM_Descrip_dict

In [12]:
# ── 6. Mouse NCBI Gene Info ───────────────────────────────────────────────────
# Official NCBI gene_info file for Mus musculus
NCBI_MOUSE_gene = pd.read_csv(f'{BASE_PATH}databases_for_mapping/ncbi/Mus_musculus.gene_info',
    sep='\t'
)

# Extract Ensembl Mouse Gene ID (ENSMUSG...) from the pipe-delimited dbXrefs column
# e.g. 'Ensembl:ENSMUSG00000051951|...' → 'ENSMUSG00000051951'
NCBI_MOUSE_gene['ENSMBL'] = NCBI_MOUSE_gene['dbXrefs'].str.extract(r'(ENSMUSG\d+)', expand=False)

# Ensembl Mouse ID → Mouse Gene Symbol  (for Ensembl-based input data)
NCBI_Mouse_gene_Locus_2_GeneSymd_ict = dict(zip(NCBI_MOUSE_gene['ENSMBL'],  NCBI_MOUSE_gene['Symbol']))

# Mouse Gene Symbol → description       (used as Head/Tail_detail_name for Mouse Gene nodes)
NCBI_Mouse_SYM_Descrip_dict = dict(zip(NCBI_MOUSE_gene['Symbol'], NCBI_MOUSE_gene['description']))


# ── 7. Gene Ontology (GO) ─────────────────────────────────────────────────────
# GO term table: GO ID, human-readable name, and namespace (domain)
GO = pd.read_csv(f'{BASE_PATH}databases_for_mapping/MESH/MESH_GO_ID_NAME.csv')

# GO ID → term name  (e.g. 'GO:0008150' → 'biological_process')
GO_dict = dict(zip(GO['id'], GO['name']))

# Normalize namespace labels to title case for display consistency in the KG
GO["namespace"] = GO["namespace"].replace({
    "biological_process": "Biological Process",
    "molecular_function": "Molecular Function",
    "cellular_component": "Cellular Component"
})

# GO ID → namespace  (used to tag GO nodes with their domain/type)
GO_namespace_dict = dict(zip(GO['id'], GO['namespace']))

# Aging bank data

# species list has incomplete species representation , head detail name not mapped correctly

# 1

In [182]:

df_file = pd.read_csv(f'{BASE_PATH}agingbank/all_experimentally supported associations.txt',sep = '\t')
Sp_list = ['Caenorhabditis elegans','Drosophila melanogaster','Saccharomyces cerevisiae','Homo sapiens','Mus musculus','Danio rerio']
df_file = df_file[df_file['Organism'].isin(Sp_list)]
df_file

print(df_file['Anti/Pro'].value_counts())
# Standardize the Anti/Pro column
df_file['Anti/Pro'] = df_file['Anti/Pro'].str.strip().str.capitalize()

# Handle 'Anti/pro' edge case → 'Anti/Pro'
df_file['Anti/Pro'] = df_file['Anti/Pro'].str.replace(r'^Anti/pro$', 'Anti/Pro', regex=True)
df_file = df_file[df_file['Anti/Pro'] != 'Anti/Pro']
df_file = df_file[~df_file['Anti/Pro'].isna()]
print(df_file['Anti/Pro'].value_counts())

# Rename columns
df_file.rename(columns={
    'name': 'Head',
    # 'ncRNA regulate gene': 'Tail'
}, inplace=True)
df_file = df_file[~df_file['Head'].isna()]
df_file['Head_type'] = 'Gene'

df_file
# Map Anti/Pro → relation
relation_map = {
    'Pro':  'Gene_promotes_BiologicalProcess',
    'Anti': 'Gene_inhibits_BiologicalProcess'
}

df_file = df_file.copy()

df_file['relation']         = df_file['Anti/Pro'].map(relation_map)
df_file['Tail']             = 'GO:0007568'
df_file['Tail_detail_name'] = 'aging'
df_file['Tail_type']        = 'BiologicalProcess'
df_file['Tail_id_is']       = 'Quick_GO'
df_file['Head_ID_IS']       = 'NCBI_ID'
df_file['kg_source']        = 'AgingBank'
df_file['Data_type']        = 'Causative'
df_file['species']          = df_file['Organism']
df_file['kg_type'] = 'Aging'
# Drop rows where relation couldn't be mapped
df_file = df_file[df_file['relation'].notna()]

print(df_file['relation'].value_counts())
df_file

Anti/Pro
Pro         1166
Anti         673
anti          61
pro           16
PRO            7
Anti/Pro       2
ANTI           1
Name: count, dtype: int64
Anti/Pro
Pro     1189
Anti     735
Name: count, dtype: int64
relation
Gene_promotes_BiologicalProcess    1178
Gene_inhibits_BiologicalProcess     734
Name: count, dtype: int64


,id,ncRNA type,Head,Organism,Experiment method,sample,tissue,Up/Down,Anti/Pro,function description,...,relation,Tail,Tail_detail_name,Tail_type,Tail_id_is,Head_ID_IS,kg_source,Data_type,species,kg_type
0,1,alternative splicing,15-LOX2sv-a,Homo sapiens,RT-PCR,cell,prostate,Up,Pro,"First, the 15-LOX2 promoter activity and the m...",...,Gene_promotes_BiologicalProcess,GO:0007568,aging,BiologicalProcess,Quick_GO,NCBI_ID,AgingBank,Causative,Homo sapiens,Aging
1,2,alternative splicing,15-LOX2sv-b,Homo sapiens,RT-PCR,cell,prostate,Up,Pro,"First, the 15-LOX2 promoter activity and the m...",...,Gene_promotes_BiologicalProcess,GO:0007568,aging,BiologicalProcess,Quick_GO,NCBI_ID,AgingBank,Causative,Homo sapiens,Aging
2,3,alternative splicing,15-LOX2sv-c,Homo sapiens,RT-PCR,cell,prostate,Up,Pro,"First, the 15-LOX2 promoter activity and the m...",...,Gene_promotes_BiologicalProcess,GO:0007568,aging,BiologicalProcess,Quick_GO,NCBI_ID,AgingBank,Causative,Homo sapiens,Aging
4,5,alternative splicing,AS-NMD,Caenorhabditis elegans,"PCR,other",animal,NaN,Down,Anti,"Due to increased AS, NMD pathway genes are upr...",...,Gene_inhibits_BiologicalProcess,GO:0007568,aging,BiologicalProcess,Quick_GO,NCBI_ID,AgingBank,Causative,Caenorhabditis elegans,Aging
8,9,alternative splicing,I-457,Mus musculus,qPCR,animal,brain,Up,Pro,I-457 expression showed age-dependent changes ...,...,Gene_promotes_BiologicalProcess,GO:0007568,aging,BiologicalProcess,Quick_GO,NCBI_ID,AgingBank,Causative,Mus musculus,Aging
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2886,2887,TF,WT1,Mus musculus,qPCR,animal,mouse lung tumor cell lines (LKR10 and LKR13),Down,Pro,Using a combination of functional genomics and...,...,Gene_promotes_BiologicalProcess,GO:0007568,aging,BiologicalProcess,Quick_GO,NCBI_ID,AgingBank,Causative,Mus musculus,Aging
2887,2888,TF,WT1,Homo sapiens,Chemiluminescence ELISA (CL-ELISA);Luciferase ...,cell line,The murine melanoma B16F10-Nex2 subline；The hu...,NaN,Pro,Peptide WT1-pTj quickly penetrated human melan...,...,Gene_promotes_BiologicalProcess,GO:0007568,aging,BiologicalProcess,Quick_GO,NCBI_ID,AgingBank,Causative,Homo sapiens,Aging
2888,2889,TF,WT1,Mus musculus,Chemiluminescence ELISA (CL-ELISA);Luciferase ...,cell line,The murine melanoma B16F10-Nex2 subline；The hu...,NaN,Pro,Peptide WT1-pTj quickly penetrated human melan...,...,Gene_promotes_BiologicalProcess,GO:0007568,aging,BiologicalProcess,Quick_GO,NCBI_ID,AgingBank,Causative,Mus musculus,Aging
2889,2890,TF,ZBP-89,Mus musculus,"PCR,other",animal,colon,Down,Pro,"ZBP-89 (Zfp148, ZNF148) is a Kruppel-type zinc...",...,Gene_promotes_BiologicalProcess,GO:0007568,aging,BiologicalProcess,Quick_GO,NCBI_ID,AgingBank,Causative,Mus musculus,Aging


In [183]:
df_file['Organism'].value_counts(dropna=False)

Organism
Homo sapiens                1130
Mus musculus                 597
Caenorhabditis elegans       131
Drosophila melanogaster       38
Saccharomyces cerevisiae      16
Name: count, dtype: int64

In [184]:
organism_dfs = {
    species: df_file[df_file['Organism'] == species].copy()
    for species in df_file['Organism'].unique()
}

# Unpack into named dataframes
agingbank_human      = organism_dfs['Homo sapiens']
agingbank_mouse      = organism_dfs['Mus musculus']
agingbank_celegans   = organism_dfs['Caenorhabditis elegans']
agingbank_drosophila = organism_dfs['Drosophila melanogaster']
agingbank_yeast      = organism_dfs['Saccharomyces cerevisiae']

# Verify
for name, df in [('Human', agingbank_human), ('Mouse', agingbank_mouse), 
                 ('C.elegans', agingbank_celegans), ('Drosophila', agingbank_drosophila), 
                 ('Yeast', agingbank_yeast)]:
    print(f"{name}: {df.shape[0]} rows")

Human: 1130 rows
Mouse: 597 rows
C.elegans: 131 rows
Drosophila: 38 rows
Yeast: 16 rows


## HUMAN

In [ ]:
agingbank_human['Head_Alt'] = agingbank_human['Head'].map(exploded_dict_NCBI_ALL_GENE_Syn_Dict)
agingbank_human['Head_Detail_name'] = agingbank_human['Head'].map(NCBI_ALL_Symb_Desc_dict).fillna(agingbank_human['Head_Alt'].map(NCBI_ALL_Symb_Desc_dict))
agingbank_human = agingbank_human[~agingbank_human['Head_Detail_name'].isna()]
agingbank_human

In [138]:
agingbank_human.columns = agingbank_human.columns.str.lower()

cols = ['head', 'relation', 'tail', 'head_type', 'tail_type',
        'kg_source', 'kg_type', 'head_detail_name', 'tail_detail_name', 
        'head_id_is', 'tail_id_is', 'kg_type', 'species']

# Keep only columns that exist in the dataframe
cols_present = [c for c in cols if c in agingbank_human.columns]

agingbank_human = agingbank_human[cols_present]

print(agingbank_human.shape)
agingbank_human_promotes = agingbank_human[agingbank_human['relation'] == 'Gene_promotes_BiologicalProcess'].copy()
agingbank_human_inhibits = agingbank_human[agingbank_human['relation'] == 'Gene_inhibits_BiologicalProcess'].copy()

print(f"Promotes: {agingbank_human_promotes.shape[0]} rows")
print(f"Inhibits: {agingbank_human_inhibits.shape[0]} rows")

agingbank_human_promotes.to_csv(f'{OUT_PATH}agingbank/Human_AgingbankGene_promotes_BioPro.csv', index=None)
agingbank_human_inhibits.to_csv(f'{OUT_PATH}agingbank/Human_AgingbankGene_inhibits_BioPro.csv', index=None)

(493, 13)
Promotes: 322 rows
Inhibits: 171 rows


## MOUSE

In [152]:
agingbank_mouse['Head_Detail_name'] = agingbank_mouse['Head'].map(NCBI_Mouse_SYM_Descrip_dict)
agingbank_mouse = agingbank_mouse[~agingbank_mouse['Head_Detail_name'].isna()]
agingbank_mouse

agingbank_mouse.columns = agingbank_mouse.columns.str.lower()

cols = ['head', 'relation', 'tail', 'head_type', 'tail_type',
        'kg_source', 'kg_type', 'head_detail_name', 'tail_detail_name', 
        'head_id_is', 'tail_id_is', 'kg_type', 'species']

# Keep only columns that exist in the dataframe
cols_present = [c for c in cols if c in agingbank_mouse.columns]

agingbank_mouse = agingbank_mouse[cols_present]

print(agingbank_mouse.shape)
agingbank_mouse_promotes = agingbank_mouse[agingbank_mouse['relation'] == 'Gene_promotes_BiologicalProcess'].copy()
agingbank_mouse_inhibits = agingbank_mouse[agingbank_mouse['relation'] == 'Gene_inhibits_BiologicalProcess'].copy()

print(f"Promotes: {agingbank_mouse_promotes.shape[0]} rows")
print(f"Inhibits: {agingbank_mouse_inhibits.shape[0]} rows")

agingbank_mouse_promotes.to_csv(f'{OUT_PATH}agingbank/Mouse_AgingbankGene_promotes_BioPro.csv', index=None)
agingbank_mouse_inhibits.to_csv(f'{OUT_PATH}agingbank/Mouse_AgingbankGene_inhibits_BioPro.csv', index=None)

(37, 13)
Promotes: 20 rows
Inhibits: 17 rows


/tmp/ipykernel_2627425/3623971736.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  agingbank_mouse['Head_Detail_name'] = agingbank_mouse['Head'].map(NCBI_Mouse_SYM_Descrip_dict)


## Celegans

In [185]:
agingbank_celegans['Head_Detail_name'] = agingbank_celegans['Head'].map(NCBI_Cele_SYM_Descrip_dict).fillna(agingbank_celegans['Head'].str.lower().map(lower_NCBI_Cele_SYM_Descrip_dict))
agingbank_celegans = agingbank_celegans[~agingbank_celegans['Head_Detail_name'].isna()]
agingbank_celegans

agingbank_celegans.columns = agingbank_celegans.columns.str.lower()

cols = ['head', 'relation', 'tail', 'head_type', 'tail_type',
        'kg_source', 'kg_type', 'head_detail_name', 'tail_detail_name', 
        'head_id_is', 'tail_id_is', 'kg_type', 'species']

# Keep only columns that exist in the dataframe
cols_present = [c for c in cols if c in agingbank_celegans.columns]

agingbank_celegans = agingbank_celegans[cols_present]

print(agingbank_celegans.shape)
agingbank_celegans_promotes = agingbank_celegans[agingbank_celegans['relation'] == 'Gene_promotes_BiologicalProcess'].copy()
agingbank_celegans_inhibits = agingbank_celegans[agingbank_celegans['relation'] == 'Gene_inhibits_BiologicalProcess'].copy()

print(f"Promotes: {agingbank_celegans_promotes.shape[0]} rows")
print(f"Inhibits: {agingbank_celegans_inhibits.shape[0]} rows")

agingbank_celegans_promotes.to_csv(f'{OUT_PATH}agingbank/Celegans_AgingbankGene_promotes_BioPro.csv', index=None)
agingbank_celegans_inhibits.to_csv(f'{OUT_PATH}agingbank/Celegans_AgingbankGene_inhibits_BioPro.csv', index=None)

(85, 13)
Promotes: 21 rows
Inhibits: 64 rows


## Drosophila

In [186]:
agingbank_drosophila['Head_Detail_name'] = agingbank_drosophila['Head'].map(NCBI_droso_SYM_Descrip_dict).fillna(agingbank_drosophila['Head'].str.lower().map(NCBI_droso_SYM_Descrip_dict))
agingbank_drosophila = agingbank_drosophila[~agingbank_drosophila['Head_Detail_name'].isna()]
agingbank_drosophila

agingbank_drosophila.columns = agingbank_drosophila.columns.str.lower()

cols = ['head', 'relation', 'tail', 'head_type', 'tail_type',
        'kg_source', 'kg_type', 'head_detail_name', 'tail_detail_name', 
        'head_id_is', 'tail_id_is', 'kg_type', 'species']

# Keep only columns that exist in the dataframe
cols_present = [c for c in cols if c in agingbank_drosophila.columns]

agingbank_drosophila = agingbank_drosophila[cols_present]

print(agingbank_drosophila.shape)
agingbank_drosophila_promotes = agingbank_drosophila[agingbank_drosophila['relation'] == 'Gene_promotes_BiologicalProcess'].copy()
agingbank_drosophila_inhibits = agingbank_drosophila[agingbank_drosophila['relation'] == 'Gene_inhibits_BiologicalProcess'].copy()

print(f"Promotes: {agingbank_drosophila_promotes.shape[0]} rows")
print(f"Inhibits: {agingbank_drosophila_inhibits.shape[0]} rows")

agingbank_drosophila_promotes.to_csv(f'{OUT_PATH}agingbank/Droso_AgingbankGene_promotes_BioPro.csv', index=None)
agingbank_drosophila_inhibits.to_csv(f'{OUT_PATH}agingbank/Droso_AgingbankGene_inhibits_BioPro.csv', index=None)

(6, 13)
Promotes: 4 rows
Inhibits: 2 rows


## yeast

In [189]:
agingbank_yeast['Head_Detail_name'] = agingbank_yeast['Head'].map(NCBI_saccharo_SYM_Descrip_dict).fillna(agingbank_yeast['Head'].str.lower().map(NCBI_saccharo_SYM_Descrip_dict))
agingbank_yeast[~agingbank_yeast['Head_Detail_name'].isna()]
# agingbank_yeast

agingbank_yeast.columns = agingbank_yeast.columns.str.lower()

cols = ['head', 'relation', 'tail', 'head_type', 'tail_type',
        'kg_source', 'kg_type', 'head_detail_name', 'tail_detail_name', 
        'head_id_is', 'tail_id_is', 'kg_type', 'species']

# Keep only columns that exist in the dataframe
cols_present = [c for c in cols if c in agingbank_yeast.columns]

agingbank_yeast = agingbank_yeast[cols_present]

print(agingbank_yeast.shape)
agingbank_yeast_promotes = agingbank_yeast[agingbank_yeast['relation'] == 'Gene_promotes_BiologicalProcess'].copy()
agingbank_yeast_inhibits = agingbank_yeast[agingbank_yeast['relation'] == 'Gene_inhibits_BiologicalProcess'].copy()

print(f"Promotes: {agingbank_yeast_promotes.shape[0]} rows")
print(f"Inhibits: {agingbank_yeast_inhibits.shape[0]} rows")

agingbank_yeast_promotes.to_csv(f'{OUT_PATH}agingbank/yeast_AgingbankGene_promotes_BioPro.csv', index=None)
agingbank_yeast_inhibits.to_csv(f'{OUT_PATH}agingbank/yeast_AgingbankGene_inhibits_BioPro.csv', index=None)

(16, 13)
Promotes: 8 rows
Inhibits: 8 rows


# 2

In [198]:
df_file =  pd.read_csv(f'{BASE_PATH}agingbank/human_experimentally supported associations.txt',sep = '\t')

print(set(df_file['Organism']))
df_file.loc[df_file['Organism'] == 'Worm', 'Organism'] = 'Caenorhabditis elegans'
df_file
Sp_list = ['Caenorhabditis elegans','Drosophila melanogaster','Saccharomyces cerevisiae','Homo sapiens','Mus musculus','Danio rerio']
df_file = df_file[df_file['Organism'].isin(Sp_list)]
print(set(df_file['Organism']))
df_file

print(df_file['Anti/Pro'].value_counts())
# Standardize the Anti/Pro column
df_file['Anti/Pro'] = df_file['Anti/Pro'].str.strip().str.capitalize()

# Handle 'Anti/pro' edge case → 'Anti/Pro'
df_file['Anti/Pro'] = df_file['Anti/Pro'].str.replace(r'^Anti/pro$', 'Anti/Pro', regex=True)
df_file = df_file[df_file['Anti/Pro'] != 'Anti/Pro']
df_file = df_file[~df_file['Anti/Pro'].isna()]
print(df_file['Anti/Pro'].value_counts())

# Rename columns
df_file.rename(columns={
    'name': 'Head',
    # 'ncRNA regulate gene': 'Tail'
}, inplace=True)
df_file = df_file[~df_file['Head'].isna()]
df_file['Head_type'] = 'Gene'

df_file
# Map Anti/Pro → relation
relation_map = {
    'Pro':  'Gene_promotes_BiologicalProcess',
    'Anti': 'Gene_inhibits_BiologicalProcess'
}

df_file = df_file.copy()

df_file['relation']         = df_file['Anti/Pro'].map(relation_map)
df_file['Tail']             = 'GO:0007568'
df_file['Tail_detail_name'] = 'aging'
df_file['Tail_type']        = 'BiologicalProcess'
df_file['Tail_id_is']       = 'Quick_GO'
df_file['Head_ID_IS']       = 'NCBI_ID'
df_file['kg_source']        = 'AgingBank'
df_file['Data_type']        = 'Causative'
df_file['species']          = df_file['Organism']
df_file['kg_type'] = 'Aging'
# Drop rows where relation couldn't be mapped
df_file = df_file[df_file['relation'].notna()]

print(df_file['relation'].value_counts())
df_file

{'Syrian hamster', 'Macaca mulatta', 'Caenorhabditis elegans', 'Rat Rattus', 'Chimpanzees', 'Homo sapiens', 'Worm', 'Mus musculus'}
{'Homo sapiens', 'Caenorhabditis elegans', 'Mus musculus'}
Anti/Pro
Pro     515
Anti    170
anti     28
pro       6
PRO       1
ANTI      1
Name: count, dtype: int64
Anti/Pro
Pro     522
Anti    199
Name: count, dtype: int64
relation
Gene_promotes_BiologicalProcess    517
Gene_inhibits_BiologicalProcess    199
Name: count, dtype: int64


,id,ncRNA type,Head,Organism,Experiment method,sample,tissue,Up/Down,Anti/Pro,function description,...,relation,Tail,Tail_detail_name,Tail_type,Tail_id_is,Head_ID_IS,kg_source,Data_type,species,kg_type
0,1,methylation,p16,Homo sapiens,qRT-PCR,animal; human,breast,Up,Anti,"In this article, we reported that disruption o...",...,Gene_inhibits_BiologicalProcess,GO:0007568,aging,BiologicalProcess,Quick_GO,NCBI_ID,AgingBank,Causative,Homo sapiens,Aging
2,3,miRNA,miR-141,Homo sapiens,"qRT-PCR,Western blot,AGO2 immunoprecipitation,",animal; human,bone,Down,Anti,The expression levels of miR-141 are positivel...,...,Gene_inhibits_BiologicalProcess,GO:0007568,aging,BiologicalProcess,Quick_GO,NCBI_ID,AgingBank,Causative,Homo sapiens,Aging
11,12,somatic mutation,SYCP2L,Homo sapiens,Western blot and immunofluorescence;RT-PCR,animal; human; cell,C57BL/6J mouse ovary and human ovary;293T cell...,NaN,Anti,Female mice lacking SYCP2L undergo a significa...,...,Gene_inhibits_BiologicalProcess,GO:0007568,aging,BiologicalProcess,Quick_GO,NCBI_ID,AgingBank,Causative,Homo sapiens,Aging
12,13,somatic mutation,SYCP2L,Mus musculus,Western blot and immunofluorescence;RT-PCR,animal; human; cell,C57BL/6J mouse ovary and human ovary;293T cell...,NaN,Anti,Female mice lacking SYCP2L undergo a significa...,...,Gene_inhibits_BiologicalProcess,GO:0007568,aging,BiologicalProcess,Quick_GO,NCBI_ID,AgingBank,Causative,Mus musculus,Aging
13,14,alternative splicing,SRSF3,Homo sapiens,"qRT-PCR,Western blot",cell line,"293T, HUVEC, NIH3T3 and MEF",Down,Pro,Down-regulated splicing factor SRSF3 is known ...,...,Gene_promotes_BiologicalProcess,GO:0007568,aging,BiologicalProcess,Quick_GO,NCBI_ID,AgingBank,Causative,Homo sapiens,Aging
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1128,1129,miRNA,miR-24,Homo sapiens,"RT-qPCR,Western blot,Luciferase reporter assay",human; cell,anterior lens capsules,Up,Pro,The oxidative stress induced upregulation of m...,...,Gene_promotes_BiologicalProcess,GO:0007568,aging,BiologicalProcess,Quick_GO,NCBI_ID,AgingBank,Causative,Homo sapiens,Aging
1129,1130,somatic mutation,p65,Homo sapiens,MTT;qRT-PCR;Western blot;Flow cytometry;Immuno...,human; cell,frozen colon cancer and matched normal adjacen...,NaN,Pro,Phosphomimetic mutation of RelA/p65 at Ser536 ...,...,Gene_promotes_BiologicalProcess,GO:0007568,aging,BiologicalProcess,Quick_GO,NCBI_ID,AgingBank,Causative,Homo sapiens,Aging
1130,1131,somatic mutation,S536D,Homo sapiens,MTT;qRT-PCR;Western blot;Flow cytometry;Immuno...,human; cell,frozen colon cancer and matched normal adjacen...,NaN,Pro,Phosphomimetic mutation of RelA/p65 at Ser536 ...,...,Gene_promotes_BiologicalProcess,GO:0007568,aging,BiologicalProcess,Quick_GO,NCBI_ID,AgingBank,Causative,Homo sapiens,Aging
1131,1132,somatic mutation,Ser536,Homo sapiens,MTT;qRT-PCR;Western blot;Flow cytometry;Immuno...,human; cell,frozen colon cancer and matched normal adjacen...,NaN,Pro,Phosphomimetic mutation of RelA/p65 at Ser536 ...,...,Gene_promotes_BiologicalProcess,GO:0007568,aging,BiologicalProcess,Quick_GO,NCBI_ID,AgingBank,Causative,Homo sapiens,Aging


In [200]:
df_file['Organism'].value_counts(dropna=False)

Organism
Homo sapiens              669
Mus musculus               45
Caenorhabditis elegans      2
Name: count, dtype: int64

In [208]:
organism_dfs = {
    species: df_file[df_file['Organism'] == species].copy()
    for species in df_file['Organism'].unique()
}

# Unpack into named dataframes
agingbank_human_2      = organism_dfs['Homo sapiens']
agingbank_mouse_2      = organism_dfs['Mus musculus']
agingbank_celegans_2   = organism_dfs['Caenorhabditis elegans']



# Verify
for name, df in [('Human', agingbank_human_2), ('Mouse', agingbank_mouse_2), 
                 ('C.elegans', agingbank_celegans_2)]:
    print(f"{name}: {df.shape[0]} rows")

Human: 669 rows
Mouse: 45 rows
C.elegans: 2 rows


## human

In [209]:
agingbank_human_2['Head_Alt'] = agingbank_human_2['Head'].map(exploded_dict_NCBI_ALL_GENE_Syn_Dict)
agingbank_human_2['Head_Detail_name'] = agingbank_human_2['Head'].map(NCBI_ALL_Symb_Desc_dict).fillna(agingbank_human_2['Head_Alt'].map(NCBI_ALL_Symb_Desc_dict))
agingbank_human_2 = agingbank_human_2[~agingbank_human_2['Head_Detail_name'].isna()]
agingbank_human_2

,id,ncRNA type,Head,Organism,Experiment method,sample,tissue,Up/Down,Anti/Pro,function description,...,Tail_detail_name,Tail_type,Tail_id_is,Head_ID_IS,kg_source,Data_type,species,kg_type,Head_Alt,Head_Detail_name
0,1,methylation,p16,Homo sapiens,qRT-PCR,animal; human,breast,Up,Anti,"In this article, we reported that disruption o...",...,aging,BiologicalProcess,Quick_GO,NCBI_ID,AgingBank,Causative,Homo sapiens,Aging,H3P10,H3 histone pseudogene 10
11,12,somatic mutation,SYCP2L,Homo sapiens,Western blot and immunofluorescence;RT-PCR,animal; human; cell,C57BL/6J mouse ovary and human ovary;293T cell...,NaN,Anti,Female mice lacking SYCP2L undergo a significa...,...,aging,BiologicalProcess,Quick_GO,NCBI_ID,AgingBank,Causative,Homo sapiens,Aging,NaN,synaptonemal complex protein 2 like
13,14,alternative splicing,SRSF3,Homo sapiens,"qRT-PCR,Western blot",cell line,"293T, HUVEC, NIH3T3 and MEF",Down,Pro,Down-regulated splicing factor SRSF3 is known ...,...,aging,BiologicalProcess,Quick_GO,NCBI_ID,AgingBank,Causative,Homo sapiens,Aging,NaN,serine and arginine rich splicing factor 3
15,16,enhancer,MEF2A,Homo sapiens,qPCR,cell line,HCAECs,na,Pro,Silencing of MEF2A in HCAECs accelerated cell ...,...,aging,BiologicalProcess,Quick_GO,NCBI_ID,AgingBank,Causative,Homo sapiens,Aging,NaN,myocyte enhancer factor 2A
16,17,enhancer,p21,Homo sapiens,Western blotting;Real-time quantitative PCR; I...,cell line,Human ES cell line (H9) with a stable normal (...,Up,Pro,MPTP-treated NPCs were found to undergo premat...,...,aging,BiologicalProcess,Quick_GO,NCBI_ID,AgingBank,Causative,Homo sapiens,Aging,H3P16,H3 histone pseudogene 16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1124,1125,somatic mutation,LMNA,Homo sapiens,"Western blotting,Immunofluorescence microscopy...",human; animal,"hTERT-TetOff-Pro,hMSCs",Up,Pro,The premature-ageing disease Hutchinson-Gilfor...,...,aging,BiologicalProcess,Quick_GO,NCBI_ID,AgingBank,Causative,Homo sapiens,Aging,NaN,lamin A/C
1125,1126,somatic mutation,OCTN2,Homo sapiens,PCR,human; animal,NaN,Up,Pro,The present study and our previous work sugges...,...,aging,BiologicalProcess,Quick_GO,NCBI_ID,AgingBank,Causative,Homo sapiens,Aging,SLC22A5,solute carrier family 22 member 5
1127,1128,methylation,ELOVL2,Homo sapiens,qRT-pCR,human; cell,blood,Up,Pro,"Using epigenomic data from public resources, w...",...,aging,BiologicalProcess,Quick_GO,NCBI_ID,AgingBank,Causative,Homo sapiens,Aging,NaN,ELOVL fatty acid elongase 2
1129,1130,somatic mutation,p65,Homo sapiens,MTT;qRT-PCR;Western blot;Flow cytometry;Immuno...,human; cell,frozen colon cancer and matched normal adjacen...,NaN,Pro,Phosphomimetic mutation of RelA/p65 at Ser536 ...,...,aging,BiologicalProcess,Quick_GO,NCBI_ID,AgingBank,Causative,Homo sapiens,Aging,WNK1,WNK lysine deficient protein kinase 1


In [210]:
agingbank_human_2.columns = agingbank_human_2.columns.str.lower()

cols = ['head', 'relation', 'tail', 'head_type', 'tail_type',
        'kg_source', 'kg_type', 'head_detail_name', 'tail_detail_name', 
        'head_id_is', 'tail_id_is', 'kg_type', 'species']

# Keep only columns that exist in the dataframe
cols_present = [c for c in cols if c in agingbank_human_2.columns]

agingbank_human_2 = agingbank_human_2[cols_present]

print(agingbank_human_2.shape)
agingbank_human_promotes_2 = agingbank_human_2[agingbank_human_2['relation'] == 'Gene_promotes_BiologicalProcess'].copy()
agingbank_human_inhibits_2 = agingbank_human_2[agingbank_human_2['relation'] == 'Gene_inhibits_BiologicalProcess'].copy()

print(f"Promotes: {agingbank_human_promotes_2.shape[0]} rows")
print(f"Inhibits: {agingbank_human_inhibits_2.shape[0]} rows")

agingbank_human_promotes_2.to_csv(f'{OUT_PATH}agingbank/Human_AgingbankGene_promotes_BioPro_file2.csv', index=None)
agingbank_human_inhibits_2.to_csv(f'{OUT_PATH}agingbank/Human_AgingbankGene_inhibits_BioPro_file2.csv', index=None)

(277, 13)
Promotes: 185 rows
Inhibits: 92 rows


## Mouse

In [211]:
agingbank_mouse_2['Head_Detail_name'] = agingbank_mouse_2['Head'].map(NCBI_Mouse_SYM_Descrip_dict)
agingbank_mouse_2 = agingbank_mouse_2[~agingbank_mouse_2['Head_Detail_name'].isna()]
agingbank_mouse_2

agingbank_mouse_2.columns = agingbank_mouse_2.columns.str.lower()

cols = ['head', 'relation', 'tail', 'head_type', 'tail_type',
        'kg_source', 'kg_type', 'head_detail_name', 'tail_detail_name', 
        'head_id_is', 'tail_id_is', 'kg_type', 'species']

# Keep only columns that exist in the dataframe
cols_present = [c for c in cols if c in agingbank_mouse_2.columns]

agingbank_mouse_2 = agingbank_mouse_2[cols_present]

print(agingbank_mouse_2.shape)
agingbank_mouse_2_promotes = agingbank_mouse_2[agingbank_mouse_2['relation'] == 'Gene_promotes_BiologicalProcess'].copy()
agingbank_mouse_2_inhibits = agingbank_mouse_2[agingbank_mouse_2['relation'] == 'Gene_inhibits_BiologicalProcess'].copy()

print(f"Promotes: {agingbank_mouse_2_promotes.shape[0]} rows")
print(f"Inhibits: {agingbank_mouse_2_inhibits.shape[0]} rows")

agingbank_mouse_2_promotes.to_csv(f'{OUT_PATH}agingbank/Mouse_AgingbankGene_promotes_BioPro_2.csv', index=None)
agingbank_mouse_2_inhibits.to_csv(f'{OUT_PATH}agingbank/Mouse_AgingbankGene_inhibits_BioPro_2.csv', index=None)

(1, 13)
Promotes: 0 rows
Inhibits: 1 rows
